In [ ]:
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/cta0001.wav
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/cta0001.seg_B1
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/cta0001_cyr_1251.seg_B1
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/d01_s01_cards.seg_R1
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/d01_s01_cards.txt
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/cta0001_new_fs.wav


### Устройство файлов SEG

#### Уровни разметки

Программа Wave Assistant использует свой стандарт для разметки аудиоданных, т.&nbsp;е. данных о начале и конце звуковых отрезков и их названиях. Wave Assistant поддерживает до 16 уровней разметки, которые кодируются цветом (зелёный &mdash; G, синий &mdash; B, красный &mdash; R и жёлтый &mdash; Y) и цифрой от 1 до 4. Каждому уровню сопоставлено своё кодовое число, которое является степенью двойки (см. таблицу).

|Уровень|Код|
|---|---|
|G1|2<sup>0</sup> = 1|
|B1|2<sup>1</sup> = 2|
|R1|2<sup>2</sup> = 4|
|Y1|2<sup>3</sup> = 8|
|G2|2<sup>4</sup> = 16|
|B2|2<sup>5</sup> = 32|
|R2|2<sup>6</sup> = 64|
|Y2|2<sup>7</sup> = 128|
|G3|2<sup>8</sup> = 256|
|B3|2<sup>9</sup> = 512|
|R3|2<sup>10</sup> = 1024|
|Y3|2<sup>11</sup> = 2048|
|G4|2<sup>12</sup> = 4096|
|B4|2<sup>13</sup> = 8192|
|R4|2<sup>14</sup> = 16384|
|Y4|2<sup>15</sup> = 32768|

Чтобы легко построить зависимость между названиями уровней и их кодами, воспользуемся функцией `product` из встроенного модуля `itertools`. Эта функция позволяет построить декартово произведение двух итерируемых объектов: из набора четырёх цветов и четырёх цифр мы получим все возможные комбинации цвета и цифры.

In [ ]:
from itertools import product

letters = "GBRY"
nums = "1234"
levels = [ch + num for num, ch in product(nums, letters)]
print(levels)
level_codes = [2 ** i for i in range(len(levels))]

Построим словари для конвертации из названия уровня в его код и наоборот. Для этого можно воспользоваться функцией `zip()`. 

<div class="note">

Функция `zip()` позволяет итерировать сразу по нескольким итерируемым объектам, например, спискам:

In [ ]:
names_list = ["John", "Jack", "Jill"]
last_names_list = ["Smith", "Jones", "Doe"]
grades_list = [5, 4, 5]
for name, last_name, grade in zip(names_list, last_names_list, grades_list):
    print(name, last_name, grade)

</div>

Используем её совместно со словарным включением:

In [ ]:
level2code = {i: j for i, j in zip(levels, level_codes)}
code2level = {j: i for i, j in zip(levels, level_codes)}

На каждом уровне пользователь имеет возможность ставить метки, обозначающие границы интересующих его явлений. Каждая метка описывается своей позицией в файле и названием (строка, которая может быть пустой)

#### Структура файла SEG

Метки Wave Assistant хранятся в специальных файлах SEG. Метки разных уровней могут быть записаны в одном файле (тогда используется расширение *.seg*) или в разных (тогда используются расширения вида *.seg_G1*, где после подчёркивания идёт название уровня).

Файл SEG представляет собой текстовый файл, состоящий из нескольких секций. Название каждой секции пишется заглавными буквами на отдельной строке и заключается в квадратные скобки.

Первая секция называется `[PARAMETERS]` (параметры) и представляет собой заголовок файла, где записаны характеристики звукового файла, а также количество меток. Типичная структура этой секции:

Здесь обычно записываются следующие параметры: частота дискретизации (`SAMPLING_FREQ`), количество байт на отсчёт (`BYTE_PER_SAMPLE`), код записи (`CODE`), количество каналов (`N_CHANNEL`), количество меток (`N_LABEL`). Значения этих параметров представляют собой целые числа и записываются после названия параметра через знак равенства.

Отдельно стоит сказать о параметре `CODE`. В большинстве случаев он будет равен 0, однако в некоторых нечасто использующихся стандартах записи он будет принимать другие значения (см. таблицу).

|Стандарт записи|Код|
|---|---|
|Signed 32-bit|1|
|Unsigned 8-bit|2|
|&mu;-law|3|

Если стандарт не указан в таблице выше, он либо кодируется кодом 0, либо не поддерживается Wave Assistant. В частности, обратите внимание, что сигналы с вещественными значениями в программе не поддерживаются.

Следующая секция называется `[LABELS]` (метки) и содержит информацию о разметке аудиофайла. Каждая метка записывается на отдельной строчке и определяется тремя параметрами: положение метки **в байтах**, уровень метки в виде степени двойки и название метки (может быть пустым). Эти значения отделяются друг от друга запятыми. Типичная структура этой секции:

Обратите внимание, что позиция метки определяется не номером отсчёта, которому она соответствует, а номером его первого байта (нумерация начинается с нуля). Чтобы перевести номер байта в номер отсчёта (т.&nbsp;е. индекс соответствующего элемента в массиве), нужно разделить его на количество байт на отсчёт и на количество каналов.

На рисунке ниже представлена схема соответствия номеров байт и отсчётов в 16-битном стереофайле. Отсчёты левого канала обозначены буквой L, отсчёты правого &mdash; буквой R.

![image](https://phonetics-spbu.github.io/courses/python_textbook/images/bytes_to_samples.png)

После секции `[LABELS]` могут встречаться и другие секции, например, `[HISTORY]`, которая содержит историю модификаций сигнала. Однако такая информация не относится собственно к разметке и встречается очень редко.

#### Кодировка файлов SEG

Поскольку файлы SEG &mdash; это текстовые файлы, при работе с ними неизбежно встаёт вопрос кодировок. Одни и те же символы могут быть по-разному представлены в памяти компьютера. За соответствие символов и их закодированных представлений отвечают кодировки. Поскольку символов, которыми пользуется человечество, очень много (сюда входят символы, цифры и знаки препинания разных письменностей, математические знаки, эмодзи и многое другое), часто в разных случаях используются разные кодировки.

Количество символов, которые может вместить себя кодировка, определяется количеством байт, которое в ней отводится на хранение одного символа. Это значение может быть как фиксированным, так и переменным. В однобайтных кодировках может быть закодировано не более 2<sup>8</sup> = 256 различных символов.

Большинство современных кодировок имеют пересечения, потому что они основаны на кодировке ASCII. Эта кодировка содержит коды для букв латинского алфавита, основных знаков препинания и ряда служебных символов. На код одного символа в ней отводится всего 7 бит (всего 128 символов), поэтому её легко расширить до однобайтной, назначив старший бит каждого кода нулём. Тогда все коды, начинающиеся с 1, могут быть использованы для представления других символов. Так устроена, например, кодировка Windows-1251: байты, старший бит которых равен 1, используются для кодирования букв кириллицы и некоторых дополнительных символов. В кодировке Windows-1252 эти же коды используются для кодирования букв европейских языков, не входящих в основную латиницу (вроде *&ouml;* или *&eacute;*).

Чаще всего сейчас используется кодировка UTF-8, в которой могут быть представлены все символы Unicode &mdash; большой кодовой таблицы, где каждому символу присвоено уникальное число. В UTF-8 один символ может быть закодирован одним, двумя, тремя или четырьмя байтами. Одним байтом кодируются символы ASCII, поэтому эти символы выглядят одинаково в UTF-8 и в других ASCII-совместимых кодировках. Отсюда следует, что файл, содержащий только символы ASCII, будет закодирован одинаково и в UTF-8, и, например, в Windows-1251.

В реальной практике настоятельно рекомендуется использование кодировки UTF-8 как международного стандарта. Кроме того, в Windows-1251 не могут быть закодированы символы транскрипций, как, например, расширенная латиница (*&scaron;* или *&ccaron;*) или символы МФА (*&oelig;* или *&#601;*). Если же разметка состоит только из символов ASCII, то выбор между Windows-1251 и UTF-8 будет неважен.

В текстовых файлах никак не отражена кодировка, в которой они записаны, поэтому, если она заранее неизвестна, однозначно определить её в общем случае невозможно. Однако из-за того, что в UTF-8 используется переменная длина кода, не все последовательности байт будут допустимы в этой кодировке. Поэтому мы точно будем знать, что файл закодирован **не** в этой кодировке, если в нём есть недопустимые последовательности байт.

Кроме того, у UTF-8 есть вариант UTF-8-BOM (`utf-8-sig` в терминологии Python), в котором первый символ &mdash; специальный знак, т.&nbsp;н. *byte order mark* (BOM) который используется в том числе для того, чтобы указать, что используется таблица Unicode. Это символ с кодом `U+FEFF` (т.&nbsp;е. FEFF<sub>16</sub> = 65279<sub>10</sub>). В UTF-8 он кодируется последовательностью байт `EF BB BF`.

При работе с файлами SEG чаще всего придётся делать выбор между Windows-1251, UTF-8 и UTF-8-BOM (Wave Assistant поддерживает также UTF-16). Можно написать специальную функцию, которая в большинстве случаев будет автоматически определять корректную кодировку. Чтобы задать кодировку, с которой будет прочитан или записан файл, передадим её в аргумент `encoding` функции `open()`.

In [ ]:
def detect_encoding(file_path: str) -> str:
    encoding = "utf-8"
    try:
        text = open(file_path, 'r', encoding="utf-8").read()
    except UnicodeDecodeError:
        try:
            open(file_path, 'r', encoding="utf-16").read()
            encoding = "utf-16"
        except UnicodeDecodeError:
            encoding = "cp1251"
    else:
        if text.startswith("\ufeff"):  # т.н. byte order mark
            encoding = "utf-8-sig"
    return encoding

Функция считает по умолчанию, что файл закодирован в UTF-8. Если при попытке открыть файл возникло исключение `UnicodeDecodeError`, то в нём присутствуют недопустимые последовательности байт для выбранной кодировки. Дальше делается аналогичная попытка для кодировки UTF-16. Если же первый символ раскодированной последовательности &mdash; `U+FEFF` (`"\ufeff"` в формате записи символов Unicode, используемом в Python), то кодировка &mdash; UTF-8-SIG.

Понятно, что некоторые коды Windows-1251 всё же будут допустимыми в UTF-8:

In [ ]:
with open("lol.txt", "w", encoding="cp1251") as f:
    f.write("Пё")

with open("lol.txt", "r", encoding="utf-8") as f:
    print(f.read())  # Greek Capital Letter Sho ???

Код выше выводит на экран символ *&#x3f8;* (Greek Capital Letter Sho). Кириллические буквы *П* и *ё* кодируются в Windows-1251 байтами `CF` и `BF`, что соответствует коду буквы *&#x3f8;* в UTF-8. Однако такие случаи в реальной практике не встречаются: вероятность того, что файл Windows-1251 будет содержать только допустимые в UTF-8 последовательности байт, близка к нулю.

Для полноты описания также стоит заметить, что описанная выше функция ошибочно определит кодировку файла Windows-1251 как UTF-16, если этот файл содержит чётное количество символов (т.&nbsp;к. в этой кодировке на каждый символ отводится либо 2, либо 4 байта) и первые два &mdash; *юя* или *яю* (что соответстует кодам символа BOM в двух разновидностях UTF-16 &mdash; с порядком байт от старшего к младшему и от младшего к старшему соответственно). Однако ясно, что файлы SEG никогда не будут удовлетворять этому условию, т.&nbsp;к. обязательно начинаются со строки `[PARAMETERS]`. 

Если набор возможных кодировок шире, чем перечисленные выше, можно воспользоваться специализированной библиотекой [`chardet`](https://pypi.org/project/chardet/). Она использует различные средства, в том числе статистические, чтобы с высокой точностью определить кодировку текста, а также язык, на котором он написан. По [ссылке](https://chardet.readthedocs.io/en/latest/) можно ознакомиться с документацией.

Откроем текстовый файл в бинарном режиме, чтобы считать оттуда &laquo;сырые&raquo; байты, которые передадим на вход функции `detect()`. Функция вернёт нам словарь, в котором в поле `"encoding"` будет лежать название кодировки, в поле `"confidence"` &mdash; её вероятность, а в поле `"language"` &mdash; язык, на котором написан текст.

In [ ]:
import chardet

with open("cta0001.seg_B1", "rb") as f:
     print(chardet.detect(f.read()))

Обратите внимание, что для правильной интерпретации однобайтных кодировок частоты символов (а точнее, их сочетаний) в тексте должны примерно совпадать с общеязыковыми. Очевидно, что в файле SEG, названиями меток которого служат отдельные кириллические буквы, этого не произойдёт, и `chardet`, вероятно, допустит ошибку.

Например, создадим версию файла `cta0001.seg_B1` с кириллической транскрипцией (здесь и далее приведена только секция `[LABELS]`):

Запишем её в файл с кодировкой Windows-1251 и попробуем определить её с помощью `chardet`:

In [ ]:
with open("cta0001_cyr_1251.seg_B1", "rb") as f:
    print(chardet.detect(f.read()))

Как видно, алгоритм определил кодировку как ISO-8859-1 (кодировка для западноевропейских языков, близкая к Windows-1252). Если мы попытаемся прочитать файл в этой кодировке, то убедимся, что кириллические символы считались неверно:

In [ ]:
with open("cta0001_cyr_1251.seg_B1", "r", encoding="iso-8859-1") as f:
    print(f.read())

Если названиями меток служат целые слова, то вероятность правильного определения выше, но всё равно не гарантирована. Статистическим алгоритмам могут мешать как служебные части файла (параметры, позиции меток и т.&nbsp;п.), так и особенности аннотации (например, цифры, которыми в некоторых корпусах отмечается ударение).

### Чтение и запись файлов SEG

#### Чтение файлов SEG

Напишем функцию для чтения файла SEG. Она будет принимать на вход путь к файлу, а возвращать параметры в виде словаря и список меток, каждая из которых будет представлена в виде словаря с тремя полями: `"position"` &mdash; индекс соответствующего отсчёта, `"level"` &mdash; буквенно-цифровое представление уровня и "`name`" &mdash; название (текст) метки.

In [ ]:
def read_seg(filename: str, encoding: str = "utf-8-sig") -> tuple[dict, list[dict]]:
    with open(filename, encoding=encoding) as f:
        lines = [line.strip() for line in f.readlines()]

    # найдём границы секций в списке строк:
    header_start = lines.index("[PARAMETERS]") + 1
    data_start = lines.index("[LABELS]") + 1

    # прочитаем параметры
    params = {}
    for line in lines[header_start:data_start - 1]:
        key, value = line.split("=")
        params[key] = int(value)

    # прочитаем метки
    labels = []
    for line in lines[data_start:]:
        # если в строке нет запятых, значит, это не метка и метки закончились
        if line.count(",") < 2:
            break
        pos, level, name = line.split(",", maxsplit=2)
        label = {
            # чтобы перевести в отсчёты, разделим на кол-во байт на отсчёт
            # и на количество каналов
            "position": int(pos) // params["BYTE_PER_SAMPLE"] // params["N_CHANNEL"],
            "level": code2level[int(level)],
            "name": name,
        }
        labels.append(label)
    return params, labels

Обратите внимание на использование параметра `maxsplit` в методе `.split()`. Поскольку в названиях меток могут содержаться запятые, вызов метода без этого параметра приведёт к разбиению имени метки по этим запятым. Поэтому зададим `maxsplit=2`, чтобы разбиение произошло только по первым двум запятым в строке.

#### Запись файлов SEG

Напишем функцию для записи меток в файл SEG. Она будет принимать на вход словарь с параметрами и список меток в том же формате, который возвращает функция `read_seg()`, а также имя целевого файла и кодировка, в которой необходимо его записать.

In [ ]:
def write_seg(params: dict, labels: list, filename: str, encoding: str = "utf-8-sig") -> None:
    # зададим значения параметров по умолчанию
    # вы можете изменить функцию так, чтобы параметры можно было передавать как ключевые слова
    param_defaults = {
        "SAMPLING_FREQ": 44100,
        "BYTE_PER_SAMPLE": 2,
        "CODE": 0,
        "N_CHANNEL": 1,
        "N_LABEL": 0
    }
    # запишем в словарь переданные в функцию значения параметров
    params = param_defaults | params
    # количество меток определим как длину списка labels
    params["N_LABEL"] = len(labels)
    with open(filename, "w", encoding=encoding) as f:
        f.write("[PARAMETERS]\n")
        for key, value in params.items():
            f.write(f"{key}={value}\n")
        f.write("[LABELS]\n")
        for label in labels:
            label_byte = (
                params['BYTE_PER_SAMPLE'] * 
                params['N_CHANNEL'] * label['position']
            )
            f.write(f"{label_byte},")
            f.write(f"{level2code[label['level']]},")
            f.write(f"{label['name']}\n")

#### Попарная обработка меток

Каждая метка обозначает единственную точку в сигнале &mdash; чаще всего границу между двумя линейными участками (например, двумя фонами или фоном и паузой). Поэтому две соседние метки будут соответствовать границам одного такого участка. Полезно уметь обрабатывать метки парами.

Опять воспользуемся функцией `zip()` (см. выше). Если списки, которые мы передаём в `zip()`, разной длины, то по умолчанию итерация будет идти до конца самого короткого из них. Поэтому мы можем передать в `zip()` список меток и новый список, который будет содержать все метки, кроме первой (для создания такого списка воспользуемся срезами).

In [ ]:
params, labels = read_seg("cta0001.seg_B1")
for start, end in zip(labels, labels[1:]):
    print(start["name"], start["position"], end["position"])

Аналогичного эффекта можно добиться с помощью функции `pairwise()` из модуля `itertools` (Python 3.10 и новее):

In [ ]:
from itertools import pairwise

params, labels = read_seg("cta0001.seg_B1")
for start, end in pairwise(labels):
    print(start["name"], start["position"], end["position"])

### Практические задания

#### 2.1. Визуализация разметки

Напишите программу, которая рисует осциллограмму с помощью `matplotlib.pyplot` и вертикальными линиями изображает положение каждой метки в файле. Используйте функцию [`plt.vlines()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.vlines.html), которая принимает на вход положение линии по оси *x* и положение её начала и конца по оси *y*. Пример результата:

![](https://phonetics-spbu.github.io/courses/python_textbook/images/cta_labels.png)

#### 2.2. Визуализация разметки с текстом

Модифицируйте программу так, чтобы в середине каждой пары меток было помещено её название. Используйте функцию [`plt.text()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.text.html), которая принимает на вход координаты по осям *x* и *y* и строку текста. Пример результата:

![](https://phonetics-spbu.github.io/courses/python_textbook/images/cta_labels_text.png)

#### 2.3. Поиск самого длинного и самого короткого интервала

Напишите программу, которая считывает файл SEG и выводит на экран самый длинный и самый короткий интервал в нём, не считая тех, у которых пустое имя. Для каждого интервала нужно вывести его начало, конец, длину (всё в секундах с точностью до 1 мс) и название.

#### 2.4. Перенос разметки из текстового файла в файл SEG

Файл *d01_s01_cards.seg_R1* содержит границы реплик, каждая из которых называется turn_XXX, где XXX &mdash; её порядковый номер. Параллельный ему файл *d01_s01_cards.txt* содержит орфографическую расшифровку этих реплик. Каждая строка файла выглядит так:

Номер и текст реплики разделены знаком табуляции (`\t`). Создайте файл *d01_s01_cards.seg_R2*, где метка, обозначающая начало каждой реплики, будет содержать её текст (без номера). Не забудьте изменить уровень меток с R1 на R2.

#### 2.5. Изменение позиций меток под новую частоту дискретизации

Файл *cta0001_new_fs.wav* содержит тот же звук, что и *cta0001.wav*, но с другой частотой дискретизации. Напишите программу, которая определяет новую частоту дискретизации и создаёт файл *cta0001_new_fs.seg_B1*, подходящий к новой версии файла.

#### 2.6. Сдвиг меток на заданный промежуток

Напишите функцию, которая принимает на вход список меток и заданный интервал, выраженный в отсчётах (положительный или отрицательный), а возвращает список меток, где позиция каждой сдвинута на этот интервал. Позиция метки после сдвига не должна стать отрицательной. Проверьте функцию на файле *cta0001.seg_B1* и сдвиге 441 отсчёт.

#### 2.7. Деление файла WAV по меткам

Напишите программу, которая считывает файл WAV и параллельный ему файл SEG, делит файл WAV на интервалы, разграниченные метками из файла SEG, и записывает каждый фрагмент в отдельный файл, названный порядковым номером интервала и именем метки, открывающей соответствующий фрагмент. Если имя метки пустое, этот интервал необходимо пропустить.

#### 2.8. Удаление согласных

Напишите программу, которая считывает файл WAV и параллельный ему файл SEG, содержащий разметку на звуки. Любой интервал, который не содержит гласные звуки, должен стать паузой. Для этого амплитуду всех отсчётов в нём нужно сделать нулём. В корпусе CORPRES, откуда взят файл *cta0001*, для обозначения гласных фонем русского языка используются латинские буквы *i, e, a, o, u, y* и цифры от 1 до 4, показывающие степень редукции, например: *a4*.

#### 2.9. (*) Создание класса `Seg`

Напишите модуль для чтения и записи файлов SEG. В модуле должны быть реализованы два класса:
* класс `Label` с полями `position` (целое число, положение метки в отсчётах), `level` (строка, формат "буква-цифра"), `name` (строка);
* класс `Seg` с методами для чтения и записи как одиночного файла *.seg*, так и группы файлов *.seg_\*\**.